# Initial Libraies

In [2]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 66.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm

import gurobipy as gp
from gurobipy import GRB
from google.colab import userdata
import os

In [4]:
class JSPMUInstance:

    def __init__(self, name=""):
        self.name = name
        self.n_jobs = 0
        self.n_machines = 0

        # jobs[j] = [(machine, duration), ...]
        self.jobs = []

        # machine_holes[m] = [(start, end), ...]
        self.machine_holes = {}

    @staticmethod
    def read(filepath):

        instance = JSPMUInstance(Path(filepath).stem)

        with open(filepath, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # ---- header ----
        header = lines[0].split()
        instance.n_jobs = int(header[0])
        instance.n_machines = int(header[1])

        # ---- jobs ----
        for j in range(instance.n_jobs):

            tokens = list(map(int, lines[1 + j].split()))

            if len(tokens) != 2 * instance.n_machines:
                raise ValueError(
                    f"Job {j} has wrong number of entries."
                )

            ops = []

            for k in range(0, len(tokens), 2):

                machine = tokens[k]
                duration = tokens[k + 1]

                if machine < 0 or machine >= instance.n_machines:
                    raise ValueError(f"Invalid machine {machine}")

                ops.append((machine, duration))

            instance.jobs.append(ops)

        # ---- initialize empty holes ----
        for m in range(instance.n_machines):
            instance.machine_holes[m] = []

        # ---- holes section ----
        idx = 1 + instance.n_jobs

        if idx < len(lines):

            if lines[idx] != "[MACHINE_HOLES]":
                raise ValueError("Missing [MACHINE_HOLES] section")

            idx += 1

            while idx < len(lines):

                tokens = list(map(int, lines[idx].split()))

                machine = tokens[0]
                n_holes = tokens[1]

                expected = 2 + 2 * n_holes
                if len(tokens) != expected:
                    raise ValueError(
                        f"Wrong hole format for machine {machine}"
                    )

                holes = []

                pos = 2
                for _ in range(n_holes):

                    start = tokens[pos]
                    duration = tokens[pos + 1]

                    if duration <= 0:
                        raise ValueError("Hole duration must be positive")

                    holes.append((start, start + duration))

                    pos += 2

                instance.machine_holes[machine] = holes

                idx += 1

        return instance

    def __str__(self) -> str:
        print('Name:',self.name)
        print("n jobs:",self.n_jobs)
        print("n machines:",self.n_machines)
        print("jobs:")
        for j in self.jobs:
          print(j)
        print("Unavailabilities:")
        for h in self.machine_holes:
          print('Machine',h,':',self.machine_holes[h])
        return ''

# Gurobipy Solutition

In [5]:
params = {
"WLSACCESSID": userdata.get('gurobi_WLSACCESSID'),
"WLSSECRET": userdata.get('gurobi_WLSSECRET'),
"LICENSEID": int(userdata.get('gurobi_LICENSEID')),
}
env = gp.Env(params=params)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 939556
Academic license 939556 - for non-commercial use only - registered to lu___@ieseg.fr


In [6]:
def plot_schedule_gantt_gurobi(
    instance,
    start_times,
    end_times,
    makespan,
    machine_unavailability=None,
    machine_labels=None,
    title="Job Shop Schedule with Machine Unavailability",
    figsize=(14, 6),
    annotate=True,
    clip_to_makespan=True,
):
    """
    Plot a Gantt chart for a JSPMU solution obtained from gurobipy.

    Parameters
    ----------
    instance : JSPMUInstance
        Problem instance.
    start_times : dict
        Dictionary {(j, h): start_time}.
    end_times : dict
        Dictionary {(j, h): end_time}.
    makespan : int or float
        Makespan of the real jobs.
    machine_unavailability : dict or None
        Optional dictionary {m: [(start, end), ...]}.
        If None, uses instance.machine_holes.
    machine_labels : dict or None
        Optional mapping {m: label}.
    title : str
        Plot title.
    figsize : tuple
        Figure size.
    annotate : bool
        If True, write operation labels inside bars.
    clip_to_makespan : bool
        If True, plot unavailability only up to the makespan.

    Returns
    -------
    fig, ax
    """
    if machine_unavailability is None:
        machine_unavailability = instance.machine_holes

    num_machines = instance.n_machines
    cmap = matplotlib.colormaps["tab20"]
    job_colors = {j: cmap(j % 20) for j in range(instance.n_jobs)}

    fig, ax = plt.subplots(figsize=figsize)
    bar_height = 0.8

    # Plot machine unavailability
    for m in range(num_machines):
        intervals = machine_unavailability.get(m, [])
        for start, end in intervals:
            if clip_to_makespan:
                if start >= makespan:
                    continue
                end_plot = min(end, makespan)
            else:
                end_plot = end

            if end_plot <= start:
                continue

            ax.barh(
                y=m,
                width=end_plot - start,
                left=start,
                height=bar_height,
                color="lightgrey",
                edgecolor="grey",
                alpha=0.9,
                zorder=1,
            )

    # Plot scheduled job operations
    plotted_jobs = set()

    for j, job in enumerate(instance.jobs):
        for h, (m, duration_nominal) in enumerate(job):
            if (j, h) not in start_times or (j, h) not in end_times:
                continue

            s = start_times[(j, h)]
            e = end_times[(j, h)]
            duration = e - s

            ax.barh(
                y=m,
                width=duration,
                left=s,
                height=bar_height,
                color=job_colors[j],
                edgecolor="black",
                zorder=2,
            )

            if annotate:
                ax.text(
                    s + duration / 2,
                    m,
                    f"J{j}-O{h}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="black",
                    zorder=3,
                )

            plotted_jobs.add(j)

    # Axis formatting
    y_ticks = list(range(num_machines))
    if machine_labels is None:
        y_ticklabels = [f"M{m}" for m in range(num_machines)]
    else:
        y_ticklabels = [machine_labels.get(m, f"M{m}") for m in range(num_machines)]

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_ticklabels)

    ax.set_xlabel("Time")
    ax.set_ylabel("Machine")
    ax.set_title(title)
    ax.grid(axis="x", linestyle="--", alpha=0.5)
    ax.invert_yaxis()

    # Make x-axis end at makespan for a cleaner plot
    ax.set_xlim(0, makespan)

    # Legend
    job_patches = [
        mpatches.Patch(color=job_colors[j], label=f"Job {j}")
        for j in sorted(plotted_jobs)
    ]
    unavailability_patch = mpatches.Patch(color="lightgrey", label="Unavailability")

    handles = job_patches + [unavailability_patch] if plotted_jobs else [unavailability_patch]

    ax.legend(
        handles=handles,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0.0,
    )

    fig.tight_layout()
    return fig, ax

In [7]:
def solve_jspmu_makespan(instance, env, time_limit=None, mip_gap=None, threads=None, verbose=True):
    """
    Solve a Job Shop Scheduling Problem with Machine Unavailabilities (JSPMU)
    using a disjunctive MILP in Gurobi.

    Real operations come from instance.jobs.
    Machine unavailabilities are modeled as fixed dummy operations on their machine.

    Parameters
    ----------
    instance : JSPMUInstance
        Parsed instance.
    time_limit : float or None
        Optional Gurobi time limit in seconds.
    mip_gap : float or None
        Optional relative MIP gap.
    threads : int or None
        Optional number of threads.
    verbose : bool
        If False, suppress Gurobi output.

    Returns
    -------
    result : dict
        {
            "status": int,
            "status_name": str,
            "objective": float or None,
            "makespan": float or None,
            "start_times": {(j, h): start_time},
            "end_times": {(j, h): end_time},
            "machine_schedules": {
                m: [
                    {
                        "type": "job" or "hole",
                        "job": j or None,
                        "op": h or None,
                        "hole_id": u or None,
                        "start": s,
                        "end": e,
                        "duration": p
                    },
                    ...
                ]
            },
            "model": model
        }
    """

    model = gp.Model(f"JSPMU_{instance.name}",env = env)

    if not verbose:
        model.Params.OutputFlag = 0
    if time_limit is not None:
        model.Params.TimeLimit = time_limit
    if mip_gap is not None:
        model.Params.MIPGap = mip_gap
    if threads is not None:
        model.Params.Threads = threads

    # ------------------------------------------------------------------
    # Build real operations
    # ------------------------------------------------------------------
    # real_ops[(j,h)] = {"machine": m, "duration": p}
    real_ops = {}
    for j in range(instance.n_jobs):
        for h, (m, p) in enumerate(instance.jobs[j]):
            real_ops[("J", j, h)] = {
                "machine": m,
                "duration": p,
                "job": j,
                "op": h,
            }

    # ------------------------------------------------------------------
    # Build dummy operations for machine holes
    # ------------------------------------------------------------------
    # hole_ops[("U", m, u)] = {"machine": m, "duration": dur, "fixed_start": s}
    hole_ops = {}
    for m in range(instance.n_machines):
        holes = instance.machine_holes.get(m, [])
        for u, (start, end) in enumerate(holes):
            hole_ops[("U", m, u)] = {
                "machine": m,
                "duration": end - start,
                "fixed_start": start,
                "hole_id": u,
            }

    all_ops = {}
    all_ops.update(real_ops)
    all_ops.update(hole_ops)

    # ------------------------------------------------------------------
    # Horizon / Big-M
    # ------------------------------------------------------------------
    total_real_proc = sum(data["duration"] for data in real_ops.values())
    total_hole_proc = sum(data["duration"] for data in hole_ops.values())
    max_hole_end = max(
        (data["fixed_start"] + data["duration"] for data in hole_ops.values()),
        default=0
    )

    # Safe upper bound
    H = total_real_proc + total_hole_proc + max_hole_end
    M = H

    # ------------------------------------------------------------------
    # Variables
    # ------------------------------------------------------------------
    # Start times for all operations
    x = {}
    for op in all_ops:
        x[op] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=H, name=f"x_{op}")

    # Makespan of real jobs only
    Cmax = model.addVar(vtype=GRB.INTEGER, lb=0, ub=H, name="Cmax")

    # ------------------------------------------------------------------
    # Fix dummy operations to machine unavailability start dates
    # ------------------------------------------------------------------
    for op, data in hole_ops.items():
        model.addConstr(x[op] == data["fixed_start"], name=f"fix_{op}")

    # ------------------------------------------------------------------
    # Job precedence constraints for real jobs
    # ------------------------------------------------------------------
    for j in range(instance.n_jobs):
        num_ops = len(instance.jobs[j])
        for h in range(1, num_ops):
            prev_op = ("J", j, h - 1)
            curr_op = ("J", j, h)
            p_prev = real_ops[prev_op]["duration"]

            model.addConstr(
                x[curr_op] >= x[prev_op] + p_prev,
                name=f"prec_j{j}_h{h}"
            )

    # ------------------------------------------------------------------
    # Makespan constraints: only real jobs count
    # ------------------------------------------------------------------
    for j in range(instance.n_jobs):
        last_h = len(instance.jobs[j]) - 1
        last_op = ("J", j, last_h)
        p_last = real_ops[last_op]["duration"]

        model.addConstr(
            Cmax >= x[last_op] + p_last,
            name=f"cmax_j{j}"
        )

    # ------------------------------------------------------------------
    # Machine conflict constraints
    # For each machine, create disjunctive constraints for every pair of
    # operations on that machine, including dummy hole operations.
    # ------------------------------------------------------------------
    ops_by_machine = {m: [] for m in range(instance.n_machines)}
    for op, data in all_ops.items():
        ops_by_machine[data["machine"]].append(op)

    y = {}  # y[a,b] = 1 if a precedes b

    for m in range(instance.n_machines):
        ops_m = ops_by_machine[m]
        n_ops_m = len(ops_m)

        for idx1 in range(n_ops_m):
            for idx2 in range(idx1 + 1, n_ops_m):
                a = ops_m[idx1]
                b = ops_m[idx2]

                # If both are fixed holes, no need for a binary.
                # They are already fixed in time.
                if a[0] == "U" and b[0] == "U":
                    continue

                y[a, b] = model.addVar(vtype=GRB.BINARY, name=f"y_{a}_before_{b}")

                p_a = all_ops[a]["duration"]
                p_b = all_ops[b]["duration"]

                # y[a,b] = 1 => a before b
                model.addConstr(
                    x[a] + p_a <= x[b] + M * (1 - y[a, b]),
                    name=f"mach_{m}_{a}_before_{b}"
                )
                model.addConstr(
                    x[b] + p_b <= x[a] + M * y[a, b],
                    name=f"mach_{m}_{b}_before_{a}"
                )

    # ------------------------------------------------------------------
    # Objective
    # ------------------------------------------------------------------
    model.setObjective(Cmax, GRB.MINIMIZE)

    # ------------------------------------------------------------------
    # Optimize
    # ------------------------------------------------------------------
    model.optimize()

    status_name = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.INTERRUPTED: "INTERRUPTED",
    }.get(model.Status, str(model.Status))

    result = {
        "status": model.Status,
        "status_name": status_name,
        "objective": None,
        "makespan": None,
        "start_times": {},
        "end_times": {},
        "machine_schedules": {},
        "model": model,
    }

    if model.Status not in {GRB.OPTIMAL, GRB.TIME_LIMIT} or model.SolCount == 0:
        return result

    # ------------------------------------------------------------------
    # Extract real-operation solution
    # ------------------------------------------------------------------
    start_times = {}
    end_times = {}

    for op, data in real_ops.items():
        _, j, h = op
        s = int(round(x[op].X))
        e = s + data["duration"]
        start_times[(j, h)] = s
        end_times[(j, h)] = e

    result["objective"] = model.ObjVal
    result["makespan"] = int(round(Cmax.X))
    result["start_times"] = start_times
    result["end_times"] = end_times

    # ------------------------------------------------------------------
    # Build machine-wise schedule including holes
    # ------------------------------------------------------------------
    machine_schedules = {m: [] for m in range(instance.n_machines)}

    for op, data in real_ops.items():
        _, j, h = op
        s = int(round(x[op].X))
        e = s + data["duration"]
        m = data["machine"]

        machine_schedules[m].append({
            "type": "job",
            "job": j,
            "op": h,
            "hole_id": None,
            "start": s,
            "end": e,
            "duration": data["duration"],
        })

    for op, data in hole_ops.items():
        _, m, u = op
        s = data["fixed_start"]
        e = s + data["duration"]

        machine_schedules[m].append({
            "type": "hole",
            "job": None,
            "op": None,
            "hole_id": u,
            "start": s,
            "end": e,
            "duration": data["duration"],
        })

    for m in range(instance.n_machines):
        machine_schedules[m].sort(key=lambda item: (item["start"], item["end"]))

    result["machine_schedules"] = machine_schedules

    return result

# Import instances

In [8]:
import zipfile
import os

In [10]:
zip_path = "generated_jspmu_instances_litarature_2.zip"
extract_dir = "generated_jspmu_instances_litarature"

# List to store file names inside the zip
file_names = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_names = zip_ref.namelist()  # names of all files/folders inside the zip
    zip_ref.extractall(extract_dir)  # unzip everything

print("Files inside zip:")
print(file_names)

Files inside zip:
['abz5_mu_mtbf300_durlognorm35_03.jspmu', 'abz6_mu_mtbf300_durlognorm35_03.jspmu', 'ft06_mu_mtbf300_durlognorm35_03.jspmu', 'ft10_mu_mtbf300_durlognorm35_03.jspmu', 'la01_mu_mtbf300_durlognorm35_03.jspmu', 'la02_mu_mtbf300_durlognorm35_03.jspmu', 'la03_mu_mtbf300_durlognorm35_03.jspmu', 'la04_mu_mtbf300_durlognorm35_03.jspmu', 'la05_mu_mtbf300_durlognorm35_03.jspmu', 'la16_mu_mtbf300_durlognorm35_03.jspmu', 'la17_mu_mtbf300_durlognorm35_03.jspmu', 'la18_mu_mtbf300_durlognorm35_03.jspmu', 'la19_mu_mtbf300_durlognorm35_03.jspmu', 'la20_mu_mtbf300_durlognorm35_03.jspmu', 'orb01_mu_mtbf300_durlognorm35_03.jspmu', 'orb02_mu_mtbf300_durlognorm35_03.jspmu', 'orb03_mu_mtbf300_durlognorm35_03.jspmu', 'orb04_mu_mtbf300_durlognorm35_03.jspmu', 'orb05_mu_mtbf300_durlognorm35_03.jspmu', 'orb06_mu_mtbf300_durlognorm35_03.jspmu', 'orb07_mu_mtbf300_durlognorm35_03.jspmu', 'orb08_mu_mtbf300_durlognorm35_03.jspmu', 'orb09_mu_mtbf300_durlognorm35_03.jspmu', 'orb10_mu_mtbf300_durlognorm3

In [11]:
data = []
problematics = []
feasibility = []

for instance_name in tqdm(file_names):
  id_instance = instance_name.split('_')[0]
  row = {'id':id_instance}
  try:
    jspmu_instance = JSPMUInstance.read(f'{extract_dir}/{instance_name}')

    res = solve_jspmu_makespan(
        jspmu_instance,
        env = env,
        time_limit=600,
        mip_gap=0.0,
        threads=2,
        verbose=False
    )

    makespan = res["makespan"]
    row['Gurobi'] = makespan

    data.append(row)
  except:
    row['Gurobi'] = -1
    problematics.append(instance_name)
    data.append(row)

100%|██████████| 24/24 [08:33<00:00, 21.38s/it]


In [12]:
df = pd.DataFrame(data)

In [13]:
df

,id,Gurobi
0,abz5,1662
1,abz6,1217
2,ft06,-1
3,ft10,1101
4,la01,994
5,la02,744
6,la03,716
7,la04,665
8,la05,673
9,la16,1046


In [15]:
df.to_csv('Optimal_Gurobi_results.csv',index=False)